# 03 — 적합도 회귀 · 미니 MLP 1000개 (`results/fitpred`)

> **GPU 상주** — ESM 토큰까지 모든 입력을 GPU 메모리에 올려 배치를 GPU 안에서 색인으로 만든다 (VRAM 180 GB급, ESM 토큰 bf16 약 131 GiB).

유전자 결손 시 균주별 z-정규화 `fit_mean` 을 예측하고, 미니 MLP 1000개의 예측 평균을 예측값, 표준편차를 불확실도로 낸다. genus 단위 중첩 교차검증으로 성능을 평가하고 전체 샘플로 최종 모델을 훈련한다.

| 항목 | 내용 |
|---|---|
| 샘플 | `fit_mean` 이 수치인 177,317개 (필수 `-inf` · 미상 제외), 46균주 · 28 genus |
| 라벨 | 균주별 z-정규화 `fit_mean` |
| 손실 | 가지별 가중 $1 - \mathrm{CCC}$ 의 평균 |
| 조기 종료 | 검증 genus 중 `\|comb_t\| ≥ 2` 샘플의 Spearman |
| 테스트 지표 | 주 지표 `_sig` (테스트 genus 중 실측이 확실한 `\|comb_t\| ≥ 2` 샘플) · 보조 지표 `_all` (전 샘플) |
| 분류 지표 | 같은 샘플에서 $\mu$ · $y$ 를 0 기준 낮음 / 높음으로 나눈 MCC · F1 · ROC AUC 등 |

진행: 1 데이터 → 2 폴드 분할 → 3 모델 · 손실 · 지표 → 4 중첩 교차검증 → 5 최종 모델 · 새 데이터 예측

In [1]:
# !pip install torch scikit-learn h5py pandas matplotlib tqdm biopython
# !pip install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128

In [2]:
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')   # VRAM 조각화 완화 (torch 불러오기 전에 설정)
import time, numpy as np, pandas as pd, torch
from proteomefit import batching, data, features, folds, cv
from proteomefit.config import Config, setup_torch
from proteomefit.models import ds_blocks, param_summary
from proteomefit.tasks import FitTask

cfg = Config(
    out_dir   = '../results/fitpred_30',
    droprate  = 0.3,                         # 연결 벡터 · 미니 MLP 은닉층 · PMA 드롭아웃
    n_branch  = 1000, br_feat = 128,          # 미니 MLP 수 · 가지당 입력 열 수
    br_hidden = (32,),                        # 미니 MLP 은닉층별 노드 수 (길이 = 은닉층 수)
    block_p   = 0.5,                          # 가지가 각 입력 블록을 고를 확률
    bootstrap = True,                         # 가지마다 훈련 샘플을 부트스트랩해 따로 학습 (배깅)
    w_clip    = (0.3, 3.0),                   # 훈련 샘플 가중치 clip(|comb_t|) 범위
    lr        = 1e-4,                         # 학습률 (AdamW)
    min_delta = 0.002,                        # 검증 Spearman (|comb_t| >= 2) 개선 인정 하한
)                                             # 나머지 설정 = proteomefit/config.py 기본값
INPUTS = ('seq', 'esm', 'prot', 'bio', 'ctx', 'ctxp', 'ctxb')   # 전체 입력 (연결 순서)
setup_torch()

print('device:', cfg.device, f'| {cfg.inner_k} inner folds (val genus matched to test extrapolation distance) '
      '| loss per-branch weighted 1-CCC | early stop on val Spearman (|comb_t| >= 2)')
print(f'target: per-organism z-scored {cfg.label_col} | {cfg.n_branch} mini-MLPs (bootstrap per branch: {cfg.bootstrap}) | test metrics: '
      f'|comb_t| >= {cfg.t_eval:g} samples in test genus (_sig, main) + all test samples (_all)')
print('autocast:', cfg.amp_dtype, '| compile:', cfg.compile, '| all inputs GPU-resident')
if cfg.device != 'cpu':
    p = torch.cuda.get_device_properties(0)
    print(f'gpu: {p.name} | VRAM {p.total_memory / 2**30:.0f} GiB | torch {torch.__version__}')

device: cuda:0 | 3 inner folds (val genus matched to test extrapolation distance) | loss per-branch weighted 1-CCC | early stop on val Spearman (|comb_t| >= 2)
target: per-organism z-scored fit_mean | 1000 mini-MLPs (bootstrap per branch: True) | test metrics: |comb_t| >= 2 samples in test genus (_sig, main) + all test samples (_all)
autocast: torch.bfloat16 | compile: True | all inputs GPU-resident
gpu: NVIDIA B200 | VRAM 178 GiB | torch 2.8.0+cu128


## 1. 데이터

### 1.1 가변 길이 입력 — 각각 PMA로 벡터 하나로 축약

| 이름 | 형태 | 출처 | 내용 |
|---|---|---|---|
| ESM 토큰 임베딩 | `[T, 1152]` | `emb_esm_600m.h5` | ESM-C 600M 토큰별 표현 (BOS/EOS 포함) |
| 정수 인코딩 서열 | `[L]` | `seq_aa` | 0 = 패딩, 1 ~ 20 = 아미노산, 21 = 그 외. 앞 `MAX_LEN` 잔기 |

### 1.2 데이터셋 블록

| 블록 | 열 | 내용 |
|---|---|---|
| `prot` | 1152 | ProteomeLM 임베딩, 균주 단위 z-정규화 |
| `bio` | 140 | Biopython 서열 특성 (물리화학 12 · 조성 20 · N말단 20 · C말단 20 · 코돈 빈도 64 · GC 4), 균주 단위 z-정규화 |
| `ctx` · `ctxp` · `ctxb` | 68 × 3 | 문맥 특성 — ESM 평균 · ProteomeLM · 둘을 이은 유사도 공간 |

**문맥 특성** — 질의 샘플마다 46개 균주 각각에서 가장 유사한 단백질 1개를 찾고, **참조 균주**(시험 · 검증 · 자기 genus 밖)의 것만 유사도 상위 $k \in \{1, 3, 5, 10, 20\}$ 개로 요약한다. 내부 폴드마다 다시 만든다.

| 묶음 | 열 | 내용 |
|---|---|---|
| `cons_*` | 9 | 상위 $k$ 평균 코사인, 1위와 3위 · 10위의 격차, 코사인 표준편차, 참조 수 |
| `otf_*` | 25 | 참조 단백질의 적합도 · 필수 여부 요약 (원본 척도) |
| `otr_*` | 12 | 같은 요약, 균주 내 순위 척도 |
| `rbh_*` | 14 | 상호 최적 히트만으로 한 요약 |
| `para_*` | 8 | 자기 게놈 안 유사 단백질의 코사인 요약 |

In [3]:
# 라벨·메타 - 적합도 수치가 있는 샘플만 (n_all 217,468 -> n 177,317)
D    = data.load_samples(cfg.base_dir, cfg.label_col, finite_only=True)
task = FitTask(D, cfg)                                          # 라벨 y = 균주별 z-정규화 적합도 [n] · |comb_t| · 조기 종료 마스크
y    = task.y

gf = pd.Series(D.yb[D.keep]).groupby(D.org)                      # 원본 적합도의 균주별 묶음
print(f'samples:{len(D.lab):,} / {len(D.base):,} (dropped essential {int(np.isneginf(D.yb).sum()):,}, '
      f'unknown {int(np.isnan(D.yb).sum()):,})')
print(f'  raw {cfg.label_col}: per-organism mean {gf.mean().min():+.3f}~{gf.mean().max():+.3f} sd {gf.std().min():.3f}~{gf.std().max():.3f}')
print(f'  label (per-organism z): range {y.min():.2f}~{y.max():.2f} | |comb_t| >= {cfg.t_eval} {int(task.ext.sum()):,}')

/tmp/ipykernel_35785/3498578745.py:16: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:203.)
  yg   = torch.from_numpy(y).float().to(DEVICE)                 # GPU 라벨 [n]


samples:177,317 / 217,468 (dropped essential 18,037, unknown 22,114)
  raw fit_mean: per-organism mean -0.243~-0.018 sd 0.182~0.695
  label (per-organism z): range -27.29~11.15 | |comb_t| >= 2.0 17,311


In [4]:
# Biopython 특성 - 전 유전자에서 계산 -> 균주별 z-정규화 -> 사용 샘플만 [n, 140]
seq_aa_all = D.base.seq_aa.fillna('').to_numpy()
BIOF  = features.bio_features(seq_aa_all, D.base.seq_dna.fillna('').to_numpy())   # [n_all, 140]
D.BIO = features.zscore_per_org(BIOF, D.org_code_all)[D.keep]
del BIOF

# 정수 인코딩 서열 - 앞 max_len 잔기를 한 배열에 이어 붙인다 (샘플 i = seq_flat[seq_off[i]:seq_off[i+1]], 훈련 때 배열째 GPU 상주)
D.seq_flat, D.seq_off, D.seq_len, raw_len = features.encode_seqs(seq_aa_all[D.keep], cfg.max_len)
print(f'seq residues {len(D.seq_flat):,} (max {int(D.seq_len.max())}, truncated {int((raw_len > cfg.max_len).sum())})')
print('bio', D.BIO.shape)

biopython: 100%|██████████| 217468/217468 [00:37<00:00, 5774.65it/s]


seq residues 60,541,732 (max 4096, truncated 80)
bio (177317, 140)


### 직교상동체 목록

전체 217,468개 단백질 각각에 대해 46개 균주마다 코사인이 가장 높은 단백질의 번호와 값(`ORT`), 자기 게놈 안 상위 20개 코사인을 구한다. 임베딩은 차원별 표준화 후 L2 정규화해 내적 = 코사인으로 만든다. 균주 프로테옴 표현 `ORG_VEC` 도 여기서 구한다.

In [5]:
prot_raw, esm_mean_raw = data.load_embeddings(cfg.prot_h5, cfg.mean_h5, D)        # [n_all, 1152] x 2
D.PROT    = features.zscore_per_org(prot_raw, D.org_code_all)[D.keep]             # 데이터셋 prot 블록 [n, 1152]
D.ORG_VEC = pd.DataFrame(prot_raw).groupby(D.org_code_all).mean().to_numpy()      # 균주 프로테옴 표현 [n_org, 1152]

# 직교상동체 목록 3종 (ESM · ProteomeLM · 둘을 이은 공간) - 값 = (sim, idx, para)
D.ORT = features.ortholog_lists(esm_mean_raw, prot_raw, D.org_code_all, D.n_org, max(cfg.topk), cfg.sim_chunk, cfg.device)
del prot_raw, esm_mean_raw
print('ortholog lists:', {k: (v[0].shape, v[2].shape) for k, v in D.ORT.items()})

para both: 100%|██████████| 46/46 [00:00<00:00, 338.71it/s]


ortholog lists: {'esm': ((217468, 46), (217468, 20)), 'prot': ((217468, 46), (217468, 20)), 'both': ((217468, 46), (217468, 20))}


In [6]:
# ESM 토큰별 임베딩 - 사용 샘플의 전 토큰을 GPU 한 배열에 이어 붙인다 (샘플 i = esm[esm_off[i]:esm_off[i+1]], bf16)
D.esm, D.esm_off, D.esm_len = data.load_esm_tokens(cfg.esm_h5, D, cfg.device, cfg.amp_dtype, cfg.esm_chunk, cfg.gpu_reserve)
print(f'ESM tokens:{len(D.esm):,} x {D.esm.shape[1]} = {D.esm.nbytes / 2**30:.1f} GiB GPU (bf16) | '
      f'free after load {torch.cuda.mem_get_info()[0] / 2**30:.1f} GiB')
print(f'tokens per sample: min {D.esm_len.min()} med {int(np.median(D.esm_len))} max {D.esm_len.max()}')

ESM -> GPU: 100%|██████████| 107/107 [08:37<00:00,  4.84s/it]

ESM tokens:60,896,366 x 1152 = 130.7 GiB GPU (bf16) | free after load 46.9 GiB
tokens per sample: min 12 med 297 max 4098


In [7]:
D.block_dim = features.block_dims(D, cfg.seq_dim)                # 입력별 연결 벡터 폭
blocks = ds_blocks(INPUTS)                                       # 데이터셋 블록
print('dataset blocks: ' + ' + '.join(f'{b}({D.block_dim[b]})' for b in blocks) + f' = {sum(D.block_dim[b] for b in blocks)}')
print(f'organisms:{D.lab.org_id.nunique()} | genus:{D.lab.genus.nunique()}')

dataset blocks: prot(1152) + bio(140) + ctx(68) + ctxp(68) + ctxb(68) = 1496
organisms:46 | genus:28


## 2. 폴드 분할 — 외부 · 내부 모두 genus 단위

$$\text{test}_{k} = G_k, \qquad \text{val}_{k,j} = G_{v_{k,j}}, \qquad \text{train}_{k,j} = \{\, i \mid \mathrm{gid}(i) \notin \{k,\ v_{k,j}\} \,\}$$

- $G_g$ — genus $g$ 의 전 샘플
- 외부 폴드 $k$ — 테스트 genus (28개, genus 단위 리브-원-아웃 교차검증(Leave-One-Out Cross-Validation, LOOCV))
- 내부 폴드 $j$ — 검증 genus $v_{k,j}$ 하나 (`INNER_K` = 3개). 내부 폴드 모델 3개의 테스트 지표 평균이 그 genus 의 성능이다

### 검증 genus 선택 — 시험 genus와 외삽 거리가 비슷한 genus

$$\mathbf{v}_g = \frac{1}{\lvert O_g \rvert} \sum_{o \in O_g} \mathbf{u}_o, \qquad s(k, g) = \cos\big(\mathbf{v}_k - \bar{\mathbf{v}},\ \mathbf{v}_g - \bar{\mathbf{v}}\big)$$

- $\mathbf{u}_o$ — 균주 $o$ 의 ProteomeLM 임베딩 평균 (`ORG_VEC`), $\mathbf{v}_g$ — genus $g$ 균주들의 $\mathbf{u}_o$ 평균 (`GVEC`), $\bar{\mathbf{v}}$ — 전 genus 평균
$$s^{\mathrm{nn}}(g \mid X) = \max_{h \notin X \cup \{g\}} s(g, h), \qquad d(v) = \big\lvert s^{\mathrm{nn}}(v \mid \{k\}) - s^{\mathrm{nn}}(k \mid \varnothing) \big\rvert$$

- $s^{\mathrm{nn}}(g \mid X)$ — genus $g$ 와, $X$ 를 뺀 나머지 genus 중 가장 닮은 것의 유사도 = **외삽 거리** (`nn_sim`)
- $s^{\mathrm{nn}}(k \mid \varnothing)$ — 시험 genus 의 외삽 거리, $s^{\mathrm{nn}}(v \mid \{k\})$ — 내부 폴드에서 검증 genus $v$ 의 외삽 거리 ($k$ 제외)
- 후보를 $d(v)$ 오름차순으로 세워 앞의 `INNER_K` 개를 검증 genus 로 쓴다
- 검증 genus 중 `|comb_t| ≥ 2` 샘플을 조기 종료에 쓴다
- 후보 = 시험 genus 와 그 최근접 genus 를 뺀 모든 genus

In [8]:
# genus 표현 -> 중심화 코사인 유사도 -> 검증 genus 선택 (후보 = 시험 genus 와 그 최근접 genus 를 뺀 전부)
S = folds.make_splits(D.gen, D.org2genus, D.ORG_VEC, cfg.inner_k, candidate=task.val_candidate)
folds.print_splits(S, D.lab, cfg.inner_k)                        # 분할 요약 + 누출 점검 (warn 이 없으면 정상)

nested CV: 28 outer x 3 inner (genus-level) = 84 trainings / 177,317 samples

fold01 Pseudomonas     : test 45,044(10org) | val Acidovorax(+0.17, nn 0.41, 3,924) Desulfovibrio(-0.32, nn 0.42, 5,236) Marinobacter(+0.13, nn 0.42, 3,648)
fold02 Ralstonia       : test 15,070(4org) | val Azospirillum(+0.11, nn 0.75, 4,178) Magnetospirillum(-0.11, nn 0.75, 3,155) Dickeya(-0.10, nn 0.76, 10,644)
fold03 Shewanella      : test 13,401(4org) | val Sphingomonas(-0.23, nn 0.60, 3,360) Caulobacter(-0.04, nn 0.60, 3,305) Dyella(-0.41, nn 0.58, 3,611)
fold04 Dickeya         : test 10,644(3org) | val Azospirillum(-0.27, nn 0.75, 4,178) Magnetospirillum(-0.32, nn 0.75, 3,155) Ralstonia(-0.10, nn 0.69, 15,070)
fold05 Cupriavidus     : test  6,373(1org) | val Azospirillum(-0.01, nn 0.75, 4,178) Magnetospirillum(-0.13, nn 0.75, 3,155) Dickeya(-0.18, nn 0.76, 10,644)
fold06 Burkholderia    : test  5,421(1org) | val Klebsiella(+0.12, nn 0.89, 4,485) Escherichia(+0.12, nn 0.89, 3,585) Dinoroseobacter(-0.28, n

## 3. 모델 · 손실 · 지표

### 구조

```
정수 서열 [B,L] ─── 임베딩 + 위치 ─ PMA ── [B,64] ───┐
ESM 토큰 [B,T,1152] ── PMA ────────────── [B,1152] ─┼─ concat ─ 미니 MLP 1000개 ─ 가지별 예측 [B,1000] ─ 평균 μ · 표준편차 σ
데이터셋 [B,d] ──────────────────────────────────────┘
```

- PMA(Pooling by Multihead Attention) — 학습 가능한 시드 벡터 1개를 쿼리로 토큰열을 어텐션 가중 평균한다
- 미니 MLP 뱅크 — 가지마다 연결 벡터의 입력 블록(seq · esm · 데이터셋 블록)을 각각 확률 `BLOCK_P` 로 고르고, 고른 블록에 열 `BR_FEAT` 개를 균등 배분해 무작위로 뽑는다. 가지 = `[Linear → GELU → Dropout] × len(BR_HIDDEN) → Linear(1)`, 층별 노드 수 `BR_FEAT → *BR_HIDDEN → 1`

### 라벨 — 균주별 z-정규화 적합도

$$y_i = (f_i - \mu_{o_i}) / \sigma_{o_i}$$

- $f_i$ — 원본 `fit_mean`, $\mu_o$ · $\sigma_o$ — 균주 $o$ 의 `fit_mean` 평균 · 표준편차

### 손실 — 가지별 가중 $1 - \mathrm{CCC}$ 의 평균

$$\mathcal{L} = \frac{1}{K} \sum_{k=1}^{K} \Big[ 1 - \frac{2\,\mathrm{cov}_w(\hat{y}_k, y)}{\mathrm{var}_w(\hat{y}_k) + \mathrm{var}_w(y) + (\bar{\hat{y}}_{k,w} - \bar{y}_w)^2} \Big], \qquad w_i = \frac{\mathrm{clip}(\lvert t_i \rvert, 0.3, 3)}{n_{\mathrm{org}}(g_i)\, n_{\mathrm{smp}}(o_i)}$$

- $\hat{y}_k$ — 가지 $k$ 의 배치 예측, $K$ = `N_BRANCH`. 가지 간 표준편차를 예측 불확실도로 쓴다
- 부트스트랩 (`BOOTSTRAP = True`) — 훈련 시작 때 가지 $k$ 마다 훈련 샘플 $m$ 개를 복원 추출해 샘플 $i$ 의 등장 횟수 $c_{ik}$ 를 정하고, 가지 $k$ 의 손실에 가중치 $w_i c_{ik}$ 를 쓴다 (배깅)
- $\mathrm{cov}_w$ · $\mathrm{var}_w$ · $\bar{\cdot}_w$ — 배치 안에서 $w_i / \sum w$ 로 구한 가중 공분산 · 분산 · 평균
- $w_i$ — `clip(|comb_t|)` 측정 가중치 × 분류군 역크기 (genus마다 같은 비중), 훈련 샘플 평균 1

### 조기 종료 — 검증 genus 중 `|comb_t| ≥ 2` 샘플의 Spearman

균주마다 $\mu$ 와 $y$ 의 Spearman 을 구해 평균한다. `|comb_t|` 는 훈련 손실의 샘플 가중치, 조기 종료 · 테스트 주 지표의 샘플 선택에만 쓰고 모델 입력에는 넣지 않는다.

### 분류 지표 — 예측 평균을 0 기준으로 나눈 낮음(0) / 높음(1)

$$MCC = \frac{TP \cdot TN - FP \cdot FN}{\sqrt{(TP+FP)(TP+FN)(TN+FP)(TN+FN)}}$$

- 실제 = $y > 0$, 예측 = $\mu > 0$ (균주 평균보다 높음). ROC AUC · PR AUC 의 점수는 $\mu$ 이고, 한 클래스뿐이면 NaN

### 테스트 지표 — 테스트 genus 샘플, 균주별로 구해 평균

주 지표는 실측이 확실한 `|comb_t| ≥ 2` 샘플(조기 종료와 같은 기준)로 계산한다. 이 샘플 집합은 모든 모델 · ablation 에 같다.

| 열 | 내용 |
|---|---|
| `n_sig` | 실측이 확실한 샘플(`|comb_t| ≥ 2`) 수 |
| `spearman_sig` · `pearson_sig` · `ccc_sig` | 실측이 확실한 샘플의 Spearman · Pearson · CCC (**주 지표**, 정렬 기준 `spearman_sig`) |
| `spearman_all` · `pearson_all` · `ccc_all` | 테스트 genus 전 샘플의 Spearman · Pearson · CCC (보조 지표) |
| `mcc` · `f1` · `recall` · `precision` · `accuracy` · `roc_auc` · `pr_auc` · `tp` · `tn` · `fp` · `fn` + `_sig` / `_all` | 같은 두 샘플 집합의 분류 지표 |

`n_org` = 주 지표가 계산된 균주 수.

### 구현 위치

| 내용 | 모듈 |
|---|---|
| 문맥 특성 · 데이터셋 (`ctx_block` · `build_dataset`) | `proteomefit/features.py` |
| 모델 (`PMA` · `SeqBranch` · `FitNet`) | `proteomefit/models.py` |
| 배치 구성 · GPU 상주 배치 공급 · 추론 | `proteomefit/batching.py` |
| 라벨 · 손실 · 샘플 가중치 · 조기 종료 지표 · 평가 산출물 (`FitTask`) | `proteomefit/tasks.py` |
| 지표 · 그림 | `proteomefit/metrics.py` · `proteomefit/plots.py` |
| 훈련 루프 (`train_model`) | `proteomefit/train.py` |
| 중첩 교차검증 · 최종 모델 · 요약 (`run_cv` · `train_final` · `print_summary`) | `proteomefit/cv.py` |

## 4. 중첩 교차검증

내부 폴드 $(k, j)$ 마다 데이터셋 생성(시험 · 검증 genus 를 참조에서 제외, 훈련 샘플 기준 표준화) → 훈련(조기 종료) → 테스트 genus 전 샘플 예측 · 지표.

| 경로 | 산출물 |
|---|---|
| `fold##_<genus>/inner##/` | `model.pt` · `history.csv` · `history.png` · `metrics_organ.csv` (균주별 지표) · `scatter.png` (실측 - 예측 산점도, `|comb_t| ≥ 2` 샘플 · 전 샘플, 막대 = $\mu \pm 1.96\sigma$) · `confusion.png` (분류 혼동행렬, `|comb_t| ≥ 2` 샘플 · 전 샘플, 세로 = 예측 · 가로 = 실제, 실제 클래스 기준 정규화) · `pred.npz` (테스트 `mu` · `sd`) |
| `fold##_<genus>/` | `metrics_folds.csv` — 내부 폴드별 1줄 (`_sd` = 균주 간 표준편차) + mean · median · std 줄 |
| `OUT_DIR/` | `metrics_total.csv` — genus별 1줄 (내부 폴드 평균, `_sd` = 내부 폴드 간 표준편차) + genus 간 mean · median · std 줄 |

`RESUME=True` 면 산출물 7종이 모두 있는 내부 폴드는 저장된 결과를 읽는다.

In [13]:
os.makedirs(cfg.out_dir, exist_ok=True)
_m = task.build_model(INPUTS, cfg.seed)
print(f'model params {param_summary(_m)} | {task.arch_msg(_m)}')
print(f'DROPRATE={cfg.droprate} | COMPILE={cfg.compile} | {S.n_fold * cfg.inner_k:,} trainings')
del _m

model params 9.76M = PMA 5.32M + seq 0.28M + bank 4.16M | concat 2712 -> 1000 x MLP(128 -> 32 -> 1), block p=0.5
DROPRATE=0.3 | COMPILE=True | 84 trainings


In [ ]:
t_start = time.perf_counter()
res = cv.run_cv(task, D, S, cfg, INPUTS, cfg.out_dir)            # genus별 결과 [n_fold, *] (metrics_total.csv 저장)


=== fold 1/28  Pseudomonas === test 45,044 samples (10 org) | 3 cached
  inner01 val Acidovorax       (sim +0.17, nn 0.41) best@10   val_sp=0.471 | Sp_sig=0.426 MCC_sig=0.300 | Sp_all=0.127 MCC_all=0.106 | cached
  inner02 val Desulfovibrio    (sim -0.32, nn 0.42) best@42   val_sp=0.345 | Sp_sig=0.434 MCC_sig=0.342 | Sp_all=0.137 MCC_all=0.120 | cached
  inner03 val Marinobacter     (sim +0.13, nn 0.42) best@10   val_sp=0.369 | Sp_sig=0.416 MCC_sig=0.293 | Sp_all=0.125 MCC_all=0.110 | cached
  fold 1/28 Pseudomonas (0:00:00) | best@10.0±18.5 val_sp=0.395±0.067 | Sp_sig=0.425±0.009 MCC_sig=0.312±0.026 | Sp_all=0.130±0.006 MCC_all=0.112±0.007

=== fold 2/28  Ralstonia === test 15,070 samples (4 org) | 3 cached
  inner01 val Azospirillum     (sim +0.11, nn 0.75) best@13   val_sp=0.515 | Sp_sig=0.418 MCC_sig=0.305 | Sp_all=0.101 MCC_all=0.084 | cached
  inner02 val Magnetospirillum (sim -0.11, nn 0.75) best@3    val_sp=0.362 | Sp_sig=0.406 MCC_sig=0.306 | Sp_all=0.098 MCC_all=0.083 | cach

In [ ]:
cv.print_summary(task, res, S.n_fold, cfg.inner_k, time.perf_counter() - t_start)
print(f'\nsaved {cfg.out_dir}/metrics_total.csv | fold##_<genus>/metrics_folds.csv | fold##_<genus>/inner##/')


SUMMARY  28 genus x 3 inner folds | total elapsed 0:00:01
  #  genus                   n    best@        val_sp     spearman_sig    ccc_sig       mcc_sig     roc_auc_sig   spearman_all    ccc_all       mcc_all     roc_auc_all 
------------------------------------------------------------------------------------------------------------------------------------------------------------------------
  1  Pseudomonas        45,044   10.0±18.5   0.395±0.067   0.425±0.009   0.368±0.022   0.312±0.026   0.733±0.004   0.130±0.006   0.294±0.009   0.112±0.007   0.573±0.002
  2  Ralstonia          15,070   13.0±6.1    0.458±0.083   0.411±0.006   0.403±0.014   0.314±0.015   0.707±0.003   0.101±0.002   0.267±0.007   0.088±0.009   0.553±0.001
  3  Shewanella         13,401   27.0±41.9   0.368±0.247   0.380±0.008   0.317±0.002   0.258±0.009   0.701±0.005   0.148±0.006   0.301±0.014   0.172±0.009   0.628±0.002
  4  Dickeya            10,644    8.0±26.9   0.440±0.074   0.490±0.002   0.462±0.035   0.366±0.0

## 5. 최종 모델 · 새 데이터 예측

- 전체 샘플로 $\lfloor \mathrm{median}_k\, \mu_k + 0.5 \rfloor$ epoch 훈련한다. $\mu_k$ = genus $k$ 내부 폴드 best epoch 의 중앙값
- 문맥 특성은 자기 genus 만 참조에서 빼고(`build_dataset(-1, -1, ·)`), 표준화 기준은 전체 샘플이다
- `task.predict_table(net, feeder, lens, smp)` — 데이터셋이 준비된 샘플의 `mu` · `sd` 를 반환한다

| 파일 (`OUT_DIR/`) | 내용 |
|---|---|
| `model_final.pt` · `history_final.csv` · `history_final.png` | 최종 모델 |

In [ ]:
model = cv.train_final(task, D, S, cfg, INPUTS, cfg.out_dir, res)   # 전체 샘플 최종 모델 -> model_final.pt
print(f'total elapsed {cv.hms(time.perf_counter() - t_start)}')

# 새 데이터 예측 - 데이터셋 준비 후 샘플 번호 smp 의 mu · sd 표
# feeder, lens = batching.make_feeder(D, INPUTS, cfg.device, cfg.t_align)
# feeder.set_dataset(features.build_dataset(D, S, ds_blocks(INPUTS), -1, -1, np.arange(task.n), cfg.topk, cfg.t_eval, cfg.device))
# task.predict_table(model, feeder, lens, smp)

final training epochs = 13 (median of genus medians 13.0 | mean of genus means 21.7)
  found ../results/fitpred_30/model_final.pt -> skip
total elapsed 0:00:01
